# 04 · Contando o estoque — **modelo genérico + especialista treinado**

Esta é a demo que amarra a palestra inteira, porque ela mostra **as duas metades
da IA aplicada** numa cena só:

| etapa | quem faz | precisou treinar? |
|---|---|---|
| achar as garrafas na foto | modelo pronto (já conhece "garrafa") | **não** |
| dizer se está aberta ou lacrada | modelo **que você ensinou** | **sim** |

> Fala de palco: *"o modelo genérico sabe o que é uma garrafa. Ele não sabe nada
> sobre o **seu** negócio. A parte que vale dinheiro é a segunda — e ela não veio
> pronta: eu ensinei separando fotos em duas pastas."*

É exatamente a mesma ideia de **agente + skill especialista**, só que em imagem.

In [ ]:
# ── 1. instala a biblioteca e monta o Google Drive ──
%pip install -q ultralytics
from google.colab import drive
drive.mount('/content/drive')

from ultralytics import YOLO
import ultralytics, torch, os, glob
ultralytics.checks()
print("GPU disponivel:", torch.cuda.is_available())

# ── 2. a raiz de tudo, e a conferência de que ela é REAL ──────────────
#
# ARMADILHA que já custou uma sessão: a linha acima cria a variável
# `drive` (minúscula), que é o MÓDULO do Colab. Se algum caminho for
# escrito com `drive` em vez de `DRIVE`, o Python aceita numa boa e
# monta um caminho como
#     <module 'google.colab.drive' from '/usr/local/...'>/04-garrafas
# O código roda, cria pastas, exporta arquivos — tudo no disco
# temporário do Colab, que evapora quando a sessão encerra. Nada disso
# chega ao seu Drive, e não há erro nenhum na tela.
#
# A conferência abaixo transforma esse silêncio num aviso imediato.

DRIVE = "/content/drive/MyDrive/PALESTRA-IA"

if not DRIVE.startswith("/content/drive/"):
    raise SystemExit(
        "DRIVE aponta para fora do Google Drive: " + repr(DRIVE) + "\n"
        "Provavelmente algum caminho usou `drive` (o módulo) em vez de `DRIVE`.")
if not os.path.isdir("/content/drive/MyDrive"):
    raise SystemExit("O Drive não montou. Rode esta célula de novo e autorize o acesso.")

os.makedirs(DRIVE, exist_ok=True)
print("raiz no Drive:", DRIVE)
print("existe de verdade:", os.path.isdir(DRIVE))

In [ ]:
# ── ajuste de PALCO: tudo grande, porque a sala enxerga de 6 a 10 m ──
import matplotlib
matplotlib.rcParams.update({
    "figure.figsize": (16, 9),
    "figure.dpi": 110,
    "font.size": 22,
    "axes.titlesize": 30,
    "axes.labelsize": 24,
    "xtick.labelsize": 20,
    "ytick.labelsize": 20,
    "legend.fontsize": 22,
    "axes.grid": True,
    "grid.alpha": .25,
    "axes.facecolor": "#0d1117",
    "figure.facecolor": "#0d1117",
    "text.color": "#e6edf3",
    "axes.labelcolor": "#e6edf3",
    "xtick.color": "#e6edf3",
    "ytick.color": "#e6edf3",
    "axes.edgecolor": "#30363d",
    "axes.titlecolor": "#3fe0a8",
})
VERDE, VERMELHO, CINZA = "#3fe0a8", "#ff5c5c", "#7d8590"
# DRIVE não é redefinido aqui de propósito: quem define é a célula de
# setup, e uma variável de caminho com duas origens é como se perde a
# noção de onde os arquivos foram parar.
print("palco configurado")

## Etapa 1 — contar (modelo pronto, zero treino)

In [ ]:
import glob, cv2, matplotlib.pyplot as plt
from collections import Counter

detector = YOLO(f"{DRIVE}/00-pesos/yolo11n.pt")
CLASSE_GARRAFA = 39          # "bottle" no COCO

fotos = [f for f in sorted(glob.glob(f"{DRIVE}/04-garrafas/inferencia/*"))
         if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp"))]
if not fotos:
    raise SystemExit("coloque fotos em 04-garrafas/inferencia/ (rode o notebook 00)")

FOTO = fotos[0]
r = detector.predict(FOTO, conf=.25, classes=[CLASSE_GARRAFA], verbose=False)[0]
print(f"garrafas encontradas: {len(r.boxes)}")

plt.figure(); plt.imshow(cv2.cvtColor(r.plot(line_width=4), cv2.COLOR_BGR2RGB))
plt.axis("off"); plt.title(f"{len(r.boxes)} garrafas — sem treinar nada")
plt.tight_layout(); plt.show()

## Etapa 2 — ensinar o que é "aberta" e "lacrada"

Aqui não existe anotação, caixa desenhada nem ferramenta especial: **duas
pastas**. O nome da pasta é o gabarito.

Treino curto de propósito — o objetivo é a **curva**, não o recorde.

In [ ]:
from IPython.display import clear_output
import matplotlib.pyplot as plt

def treinar_mostrando(modelo, dados, epocas, imgsz=224, batch=32,
                      projeto="/content/runs", nome="ao_vivo", titulo="Aprendendo"):
    """Treina e redesenha a curva a cada epoca — o ponto alto da demo."""
    hist = {}                                  # epoca -> (perda, acuracia)

    def a_cada_epoca(trainer):
        m = getattr(trainer, "metrics", None) or {}
        acc = m.get("metrics/accuracy_top1")
        perda = float(trainer.loss.item()) if getattr(trainer, "loss", None) is not None else None
        hist[trainer.epoch + 1] = (perda, acc)   # dict: a epoca repetida sobrescreve

        eps = sorted(hist)
        perdas = [hist[e][0] for e in eps]
        accs = [(hist[e][1] or 0) * 100 for e in eps]

        clear_output(wait=True)
        fig, (a1, a2) = plt.subplots(1, 2, figsize=(20, 8))
        a1.plot(eps, perdas, lw=5, color="#ff5c5c", marker="o", ms=10)
        a1.set_title("ERRO — tem que descer"); a1.set_xlabel("época")
        a2.plot(eps, accs, lw=5, color="#3fe0a8", marker="o", ms=10)
        a2.set_ylim(0, 101)
        a2.set_title("ACERTO — tem que subir"); a2.set_xlabel("época"); a2.set_ylabel("%")
        if accs:
            a2.text(eps[-1], accs[-1], f"  {accs[-1]:.0f}%", fontsize=34,
                    color="#3fe0a8", va="center", fontweight="bold")
        fig.suptitle(f"{titulo} · época {max(eps)} de {epocas}", fontsize=34)
        plt.tight_layout(); plt.show()

    modelo.add_callback("on_fit_epoch_end", a_cada_epoca)
    r = modelo.train(data=dados, epochs=epocas, imgsz=imgsz, batch=batch,
                     project=projeto, name=nome, exist_ok=True, verbose=False, plots=True)
    print("pesos e graficos em:", r.save_dir)
    return r

In [ ]:
EPOCAS = 25          # ← mexa aqui no palco: 5 aprende pouco, 25 já separa bem

especialista = YOLO(f"{DRIVE}/00-pesos/yolo11n-cls.pt")
res = treinar_mostrando(
    especialista,
    dados=f"{DRIVE}/04-garrafas/treino",
    epocas=EPOCAS,
    titulo="Aprendendo a diferença entre aberta e lacrada",
    nome="garrafas",
)

In [ ]:
# ── guarda os pesos no Drive: treinou uma vez, usa para sempre ──
import shutil
origem = f"{res.save_dir}/weights/best.pt"
destino = f"{DRIVE}/04-garrafas/pesos/garrafas_best.pt"
shutil.copy(origem, destino)
print("especialista salvo em", destino)

## Etapa 3 — os dois juntos: o inventário

In [ ]:
# ── detector acha cada garrafa · especialista julga cada uma ──
import cv2, numpy as np, matplotlib.pyplot as plt
from collections import Counter

especialista = YOLO(f"{DRIVE}/04-garrafas/pesos/garrafas_best.pt")
img = cv2.imread(FOTO)
r = detector.predict(FOTO, conf=.25, classes=[CLASSE_GARRAFA], verbose=False)[0]

placar = Counter()
anotada = img.copy()
CORES = {"lacrada": (168, 224, 63), "aberta": (92, 92, 255)}

for caixa in r.boxes.xyxy.cpu().numpy().astype(int):
    x1, y1, x2, y2 = caixa
    recorte = img[max(0, y1):y2, max(0, x1):x2]
    if recorte.size == 0:
        continue
    p = especialista.predict(recorte, verbose=False)[0]
    rotulo = p.names[int(p.probs.top1)]
    certeza = float(p.probs.top1conf)
    placar[rotulo] += 1
    cor = CORES.get(rotulo, (200, 200, 200))
    cv2.rectangle(anotada, (x1, y1), (x2, y2), cor, 4)
    cv2.putText(anotada, f"{rotulo} {certeza:.0%}", (x1, max(28, y1 - 12)),
                cv2.FONT_HERSHEY_SIMPLEX, 1.0, cor, 3)

print("INVENTARIO")
for k, v in placar.most_common():
    print(f"  {v:>3}  {k}")

plt.figure(); plt.imshow(cv2.cvtColor(anotada, cv2.COLOR_BGR2RGB)); plt.axis("off")
plt.title(f"Estoque: {sum(placar.values())} garrafas · " +
          " · ".join(f"{v} {k}" for k, v in placar.most_common()))
plt.tight_layout(); plt.show()
cv2.imwrite(f"{DRIVE}/04-garrafas/saida/inventario.jpg", anotada)

In [ ]:
# ── o painel de estoque, em tamanho de palco ──
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(16, 7))
itens = placar.most_common()
ax.bar([i[0] for i in itens], [i[1] for i in itens],
       color=[VERDE if i[0] == "lacrada" else VERMELHO for i in itens], width=.55)
for i, (k, v) in enumerate(itens):
    ax.text(i, v + .08, str(v), ha="center", fontsize=44, fontweight="bold",
            color=VERDE if k == "lacrada" else VERMELHO)
ax.set_title(f"Inventário automático · {sum(placar.values())} garrafas")
ax.set_ylabel("unidades"); ax.set_ylim(0, max(placar.values()) * 1.25)
plt.tight_layout(); plt.show()

---

# 🔴 AO VIVO · o contador de estoque

A demo de fechamento. A câmera vira um **inventário em tempo real**: cada
garrafa que entra na cena é recortada, numerada e somada ao estoque.

Dois números diferentes, e a diferença entre eles é a demo inteira:

| número | o que é |
|---|---|
| **NA CENA** | quantas garrafas a câmera está vendo **agora** |
| **ESTOQUE** | quantas garrafas **distintas** já passaram pela câmera |

Passe uma garrafa, tire, passe outra: "na cena" volta a 1, mas o **estoque
sobe para 2**. É a diferença entre olhar e contabilizar — e é exatamente o
problema de quem faz inventário de prateleira.

> Fala de palco: *"peguem uma garrafa e mostrem para a câmera. Agora tirem e
> mostrem outra. Reparem: ele não contou duas vezes a mesma, e não esqueceu a
> primeira. Isso é a diferença entre um detector e um sistema de estoque."*

In [ ]:
# ── motor de webcam ao vivo (leia o comentário: é o truque da demo) ──
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode, b64encode
import cv2, numpy as np, PIL.Image, io, time

def iniciar_webcam(largura=640, altura=480):
    # cria o video no navegador + a camada de overlay por cima dele
    display(Javascript('''
      var video, div = null, stream, imgElement, labelElement, captureCanvas;
      var pendingResolve = null, shutdown = false;
      var LARG = %d, ALT = %d;

      function removeDom() {
        if (stream) stream.getVideoTracks()[0].stop();
        if (video) video.remove();
        if (div) div.remove();
        video = null; div = null; stream = null;
        imgElement = null; captureCanvas = null; labelElement = null;
      }

      function onAnimationFrame() {
        if (!shutdown) window.requestAnimationFrame(onAnimationFrame);
        if (pendingResolve) {
          var result = "";
          if (!shutdown) {
            captureCanvas.getContext('2d').drawImage(video, 0, 0, LARG, ALT);
            result = captureCanvas.toDataURL('image/jpeg', 0.75);
          }
          var lp = pendingResolve;
          pendingResolve = null;
          lp(result);
        }
      }

      async function criarDom() {
        if (div !== null) return stream;

        div = document.createElement('div');
        div.style.border = '2px solid #3fe0a8';
        div.style.padding = '3px';
        div.style.width = '100%%';
        div.style.maxWidth = '900px';
        div.style.borderRadius = '10px';
        document.body.appendChild(div);

        var parar = document.createElement('div');
        parar.innerHTML = '&#9632; clique aqui para encerrar';
        parar.style.cssText = 'cursor:pointer;background:#3fe0a8;color:#06231a;' +
          'font-weight:700;padding:10px 16px;border-radius:8px;text-align:center;' +
          'font-family:system-ui,sans-serif;font-size:18px';
        div.appendChild(parar);
        parar.onclick = function() { shutdown = true; };

        video = document.createElement('video');
        video.style.display = 'block';
        video.style.width = '100%%';
        video.setAttribute('playsinline', '');
        video.onclick = function() { shutdown = true; };

        stream = await navigator.mediaDevices.getUserMedia(
          {video: {width: LARG, height: ALT}});
        div.appendChild(video);
        video.srcObject = stream;
        await video.play();

        // a camada que recebe o resultado do modelo, por cima do vídeo
        imgElement = document.createElement('img');
        imgElement.style.position = 'absolute';
        imgElement.style.zIndex = 1;
        imgElement.style.pointerEvents = 'none';
        imgElement.onclick = function() { shutdown = true; };
        div.appendChild(imgElement);

        labelElement = document.createElement('div');
        labelElement.style.cssText = 'font-family:system-ui,sans-serif;' +
          'font-size:20px;color:#e6edf3;padding:8px 4px';
        div.appendChild(labelElement);

        captureCanvas = document.createElement('canvas');
        captureCanvas.width = LARG;
        captureCanvas.height = ALT;
        window.requestAnimationFrame(onAnimationFrame);

        google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
        return stream;
      }

      async function quadro(rotulo, overlay) {
        if (shutdown) { removeDom(); shutdown = false; return ''; }
        stream = await criarDom();
        if (rotulo != "") labelElement.innerHTML = rotulo;
        if (overlay != "") {
          var r = video.getClientRects()[0];
          imgElement.style.top = r.top + "px";
          imgElement.style.left = r.left + "px";
          imgElement.style.width = r.width + "px";
          imgElement.style.height = r.height + "px";
          imgElement.src = overlay;
        }
        var result = await new Promise(function(resolve) { pendingResolve = resolve; });
        shutdown = false;
        return {'img': result};
      }
    ''' % (largura, altura)))


def _para_imagem(resposta):
    # base64 do navegador -> imagem BGR do OpenCV
    if not resposta:
        return None
    dados = b64decode(resposta.split(',')[1])
    arr = np.frombuffer(dados, dtype=np.uint8)
    return cv2.imdecode(arr, flags=1)


def _para_overlay(rgba):
    # array RGBA -> data URI PNG, para o navegador sobrepor ao video
    img = PIL.Image.fromarray(rgba, 'RGBA')
    buf = io.BytesIO()
    img.save(buf, format='png')
    return 'data:image/png;base64,' + b64encode(buf.getvalue()).decode('utf-8')


def rodar_ao_vivo(processa, largura=640, altura=480, rotulo_inicial='iniciando…'):
    # Laco principal.
    #
    # `processa(frame_bgr, overlay_rgba)` recebe o quadro e uma tela RGBA
    # transparente do mesmo tamanho, desenha nela, e devolve o texto do
    # painel. Encerre clicando no botão verde (ou no próprio vídeo).
    iniciar_webcam(largura, altura)
    overlay = np.zeros([altura, largura, 4], dtype=np.uint8)
    envio = ''
    rotulo = rotulo_inicial
    n = 0
    t0 = time.time()
    try:
        while True:
            resposta = eval_js('quadro("{}", "{}")'.format(rotulo, envio))
            if not resposta:
                break
            frame = _para_imagem(resposta['img'])
            if frame is None:
                break

            overlay[:] = 0
            texto = processa(frame, overlay)

            n += 1
            fps = n / max(1e-6, time.time() - t0)
            rotulo = '{} &nbsp;·&nbsp; {:.1f} quadros/s'.format(texto, fps)
            envio = _para_overlay(overlay)
    except Exception as e:
        print('encerrado:', type(e).__name__, e)
    print('fim · {} quadros processados'.format(n))

In [ ]:
# ── o laço ao vivo do estoque ────────────────────────────────────────
import cv2, numpy as np

seg_estoque = YOLO(f"{DRIVE}/00-pesos/yolo11n-seg.pt")
CLASSE_GARRAFA = 39

estoque_ids = set()        # ids distintos ja vistos = o estoque acumulado

def cor_garrafa(i):
    matiz = int((i * 53) % 180)
    b, g, r = cv2.cvtColor(np.uint8([[[matiz, 200, 255]]]), cv2.COLOR_HSV2BGR)[0][0]
    return int(r), int(g), int(b)

def processa_estoque(frame, overlay):
    h, w = frame.shape[:2]
    r = seg_estoque.track(frame, persist=True, conf=.35, classes=[CLASSE_GARRAFA],
                          verbose=False, imgsz=480)[0]

    na_cena = 0
    novas = 0
    if r.boxes is not None and len(r.boxes):
        na_cena = len(r.boxes)
        ids = (r.boxes.id.cpu().numpy().astype(int) if r.boxes.id is not None
               else np.arange(na_cena))
        mascaras = (r.masks.data.cpu().numpy() if r.masks is not None
                    else [None] * na_cena)

        for gid, mask, caixa in zip(ids, mascaras, r.boxes.xyxy.cpu().numpy()):
            gid = int(gid)
            if gid not in estoque_ids:
                estoque_ids.add(gid)
                novas += 1
            cor = cor_garrafa(gid)

            if mask is not None:
                m = cv2.resize(mask, (w, h), interpolation=cv2.INTER_NEAREST)
                mb = m.astype(bool)
                overlay[mb] = (cor[0], cor[1], cor[2], 105)
                cont, _ = cv2.findContours(m.astype(np.uint8),
                                           cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                cv2.drawContours(overlay, cont, -1, (*cor, 255), 3)

            x1, y1 = int(caixa[0]), int(caixa[1])
            cv2.rectangle(overlay, (x1, max(0, y1 - 32)), (x1 + 80, y1), (*cor, 235), -1)
            cv2.putText(overlay, "#{}".format(gid), (x1 + 8, max(20, y1 - 8)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.85, (10, 15, 20, 255), 2)

    # painel: o numero grande e o ESTOQUE, nao o "na cena"
    cv2.rectangle(overlay, (0, 0), (w, 96), (13, 17, 23, 215), -1)
    cv2.putText(overlay, "ESTOQUE", (16, 32), cv2.FONT_HERSHEY_SIMPLEX,
                0.7, (125, 133, 144, 255), 2)
    cv2.putText(overlay, str(len(estoque_ids)), (16, 86),
                cv2.FONT_HERSHEY_SIMPLEX, 2.0, (63, 224, 168, 255), 4)
    cv2.putText(overlay, "NA CENA AGORA", (int(w * .42), 32),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (125, 133, 144, 255), 2)
    cv2.putText(overlay, str(na_cena), (int(w * .42), 86),
                cv2.FONT_HERSHEY_SIMPLEX, 2.0, (230, 237, 243, 255), 4)
    if novas:
        cv2.putText(overlay, "+{}".format(novas), (int(w * .78), 86),
                    cv2.FONT_HERSHEY_SIMPLEX, 2.0, (255, 210, 90, 255), 4)

    return ("estoque <b>{}</b> &nbsp;·&nbsp; na cena <b>{}</b>"
            .format(len(estoque_ids), na_cena))

estoque_ids.clear()
rodar_ao_vivo(processa_estoque, largura=640, altura=480,
              rotulo_inicial='mostre uma garrafa para a câmera…')

In [ ]:
estoque_ids.clear()
print("estoque zerado — rode o laço de novo para recomeçar a contagem")

### O limite honesto desta demo — e como contorná-lo

O rastreamento perde o id quando a garrafa **sai do quadro e volta**: ela vira
uma garrafa nova e o estoque sobe de novo. Isso é correto para "quantas
passaram por aqui" (contador de fluxo) e **errado** para "quantas existem na
prateleira".

Para inventário de prateleira de verdade, o caminho é o oposto: uma **foto
única** da prateleira e contagem sobre ela — que é exatamente o que as células
lá de cima fazem. Cada abordagem serve a uma pergunta:

| pergunta | técnica |
|---|---|
| "quantas tem nesta prateleira?" | foto + detecção (sem rastreamento) |
| "quantas passaram pela esteira?" | vídeo + rastreamento por id |

> Se a plateia levantar esse ponto, **é ótimo**: significa que entenderam a
> diferença. A resposta honesta impressiona mais do que fingir que não existe.